## FinRED Data Set

The data set was obtained from https://github.com/soummyaah/FinRED/

Here we are doing relationship extraction. This means when we are focusing on a document, it should contain the triplet : i.e entity1 , entity2 and relationship betweem them. For that the author has taken data from two sources: Webhouse financial News and second is ECT (earning call transcripts) from seekingalpha.com (June 2019 to September 2019). They used the presentation as well as the questionnaire portion of the transcript and remove monologues with less than 200 characters. In the ECT corpus, we have about 200K monologues, 1.8M sentences with an average of 7.19 sentences per monologue. In total, we have 152K tokens in the corpus.

Wikidata KB is used for the relational triples.

In total we obtain about 21,000 sentences using the distance supervision method, however a lot of these sentences are noisy. Eg: “Delhi-based National Housing Bank (NHB) is working
to set up more than 80 new housing finance companies (HFCs), with a special emphasis on those that will focus on financing affordable houses." where the tuple is (More Than, industry, finance) since in Wikidata, “More Than“ is listed as an insurance company of the financial industry. We study a small representation of the data in detail and remove datapoints with incorrect entities such as “More Than“ and “industry“. After reduction, we get 7,775 sentences which we divide into the train, dev, and test dataset. We observe that this dataset only contains 920 sentences from the earnings call transcript and we attribute this to the earning call transcripts containing primarily conversational data which often does not contain a triplet in a sentence



In [1]:
import os
import pandas as pd
import numpy as np

##  Preamble

In [2]:
input_folder_path = '/Users/monilshah/Documents/02_NWU/10_MSDS_453_NLP/98_project_work/01_Input/01_FinRED_data/'
test_data_path   = input_folder_path + 'test.txt'
save_json_at    =  '/Users/monilshah/Documents/02_NWU/10_MSDS_453_NLP/98_project_work/02_wip_data/'

## Load Data    

In [3]:
f  = open(test_data_path,'r')
train_data = f.readlines()
len(train_data)

1068

The structure of the data is such that we have a news line and there is tuple/triplet/trio of entities identified and relationship between them extracted. However, from the same document or news headline, it is possible to extract more than one triplets. Hence, one thought process is to not have the extraction strcutured in relational data table. 

I believe JSON would be a good way to store the information. 


In [4]:
import json

# Sample line
line = 'NEW YORK (Reuters) - Apple Inc Chief Executive Steve Jobs sought to soothe investor concerns about his health on Monday, saying his weight loss was caused by a hormone imbalance that is relatively simple to treat. | Apple Inc ; Steve Jobs ; founded_by | Apple Inc ; Steve Jobs ; chief_executive_officer\n'

data_entries = []
count_skipped = 0

for line in train_data:
    # Step 1: Extract news line and entity relation.
    news_line, *entity_relations = line.strip().split(' | ')

    # Step 2: Parse each entity relationship into a dictionary format
    triples = []
    for entity_relation in entity_relations:
        try:
            # Attempt to unpack and strip
            subject, obj, relationship = map(str.strip, entity_relation.split(';'))
            triples.append({
                "subject": subject,
                "object": obj,
                "relation": relationship
            })
        except ValueError:
            # Skip if it doesn't unpack into exactly 3 values
            print(f'skiped following line : {line}')
            count_skipped += 1
            continue

    # Step 3: Create a dictionary with the news line and its corresponding triples
    if triples:  # Only add entries that have valid triples
        data_entry = {
            "news_line": news_line,
            "triples": triples
        }
        # Append to the list of entries
        data_entries.append(data_entry)


# Step 4: Save to JSON format
with open(save_json_at + 'relations_test_data.json', 'w') as file:
    json.dump([data_entries], file, indent=4)
    
print(f'Number of lines skipped : {count_skipped}')


skiped following line : Thu, Jul 30, 2015, 23:31 BST - UK Markets closed Google quietly releases new Google Glass By Andrew Trotman | Telegraph – 38 minutes ago 66.25 -0.30 | Google Glass ; Google ; developer

skiped following line : ​Mattel posts quarterly loss as Barbie sales fall Jul 17, 2015, 2:44pm PDT Share Daniel Acker | Bloomberg Mattel Inc. Barbie brand dolls are arranged for a photograph. | Mattel ; Barbie ; owner_of

skiped following line : HT Correspondent, Hindustan Times , New Delhi | Updated: Aug 01, 2015 03:10 IST Vehicles queuing up near a New Delhi petrol pump. | Hindustan Times ; New Delhi ; headquarters_location

skiped following line : Stratasys Ltd. 09/09/2015 | Press release HASCO Combines Stratasys 3D Printing with Quick-Change Mold System to Create New Price/Performance Benchmark for Low Volume Injection Molding | Stratasys ; 3D printing ; product_or_material_produced | Stratasys ; 3D printing ; industry

skiped following line : Moneylife » Pan-India number por

Some 90 lines are getting skipped due to being in non-standard format out of 5700. We will not bother about that.

There are joint entity and relation extraction modules : SPN , TPLinker ,and CasRel 

Lets try to understand how SPN works

Structured Prediction Network (SPN) is a neural network approach designed for tasks like relation extraction, where the goal is to identify relationships between entities in text. SPN works by learning to predict relationships in a structured way, often by modeling both entities and their relationships jointly within a single network. Here’s a high-level overview of how SPN-based relation extraction typically works:

Key Steps in SPN-Based Relation Extraction

Input Representation and Encoding:

Entity and Sentence Encoding: The input text, typically a sentence, is tokenized and converted into embeddings (e.g., Word2Vec, GloVe, or contextual embeddings like BERT). Entity mentions within the text are marked, allowing the model to focus on specific entity pairs.
Position Embeddings: The network also adds position embeddings, which encode the relative position of each word to the target entities, helping the model learn spatial relationships.
Neural Network Architecture:

SPNs often use bidirectional LSTMs (BiLSTMs), CNNs, or Transformers to encode the sentence context. By capturing contextual information, these networks provide a better understanding of the semantic relationships between entities.
The encoded sentence and entity information is passed through several layers to produce a rich representation of the text and entities, highlighting relationships.
Joint Entity and Relation Modeling:

SPNs use structured prediction techniques to handle both entity recognition and relation classification simultaneously. Rather than treating entities and relations separately, SPNs predict relations between pairs of recognized entities within the context.
This joint approach reduces error propagation between entity recognition and relation extraction, as both tasks benefit from shared context representations.
Attention Mechanisms:

Some SPNs incorporate attention mechanisms to weigh the importance of words around the entities, allowing the model to focus on parts of the sentence most relevant to the relationship.
This is particularly useful in long or complex sentences where relevant relational information might not be located directly between the two entities.
Structured Prediction Layer:

The structured prediction layer generates a score or probability distribution over possible relationships between entities. This is usually done through a classification layer that considers the context vector from the previous steps.
Some SPNs may also include constraints or dependencies to ensure that predicted relationships are consistent across the sentence.
Training and Loss Function:

SPNs use a loss function suited for multi-task learning, where both entities and relationships are predicted. This could be a cross-entropy loss for relation classification, combined with a loss function for entity recognition (e.g., span classification).
SPNs are typically trained on annotated corpora with labeled entities and relationships, allowing the network to learn patterns in relation context and structure.
Example of SPN in Action
Consider a sentence like:

"Apple Inc. CEO Steve Jobs introduced the iPhone."

In this example:

Entity Recognition identifies "Apple Inc." as an organization and "Steve Jobs" as a person.
Relation Extraction then tries to identify the relationship between "Apple Inc." and "Steve Jobs," which might be labeled as "CEO_of" or "founder_of."
The SPN jointly encodes both entities and the context in which they appear, focusing on the "introduced" action and the structure of the sentence to determine the specific relation.

Advantages of SPNs for Relation Extraction
Joint Entity and Relation Extraction: SPNs model entities and their relationships together, which improves accuracy over models that treat these tasks separately.
Reduced Error Propagation: Since both tasks are handled together, SPNs reduce the issue where errors in entity recognition lead to incorrect relation extraction.
Flexibility with Context and Structure: The SPN’s use of structured context encoding helps capture complex relational information, making it suitable for various types of entity relationships.
Practical Considerations
SPNs and similar architectures often rely on large labeled datasets for training, especially if using deep contextual models like BERT. Fine-tuning on domain-specific data can improve performance for specialized relation extraction tasks, such as extracting medical relations or financial connections.

In summary, SPNs are an effective, context-aware architecture for relation extraction that leverages joint entity and relationship prediction within a structured framework, often using LSTMs, CNNs, or Transformers combined with attention mechanisms to capture complex dependencies and improve accuracy.